# <font color='blue'> Chapter 32: Scaled Dot-Product Self-Attention </font>

In the previous chapter, we introduced the concepts of **Queries (Q)**, **Keys (K)**, and **Values (V)**. Every token in a sequence generates these three learned representations.

The next question is

> **How are these vectors actually used to compute attention?**

The answer is **Scaled Dot-Product Attention**, the core computation of the Transformer architecture.

This chapter derives the famous attention equation step by step, explaining the purpose of every mathematical operation and demonstrating how each token gathers information from the rest of the sequence.

---

# <font color='orange'> 1. Motivation </font>

Suppose we have the sentence

```
The

black

cat

sat
```

When processing the word

```
cat
```

the model asks

> Which other words are relevant for understanding "cat"?

Possible answers might be

```
The

↓

Small Importance

Black

↓

High Importance

Cat

↓

Highest Importance

Sat

↓

Moderate Importance
```

Self-attention computes these importance values automatically.

---

# <font color='orange'> 2. Step 1 — Compute Queries, Keys and Values </font>

Suppose

the embedding matrix is

$$
X.
$$

The model computes

$$
\boxed{
Q=XW_Q,
}
$$

$$
\boxed{
K=XW_K,
}
$$

$$
\boxed{
V=XW_V.
}
$$

These three matrices contain

- Queries,
- Keys,
- Values,

for every token in the sequence.

---

# <font color='orange'> 3. Step 2 — Compare Queries with Keys </font>

The first task is determining

how strongly every token should attend to every other token.

This is achieved by computing

$$
\boxed{
QK^T.
}
$$

Suppose

there are

four tokens.

The resulting matrix is

```
        Token1 Token2 Token3 Token4

Token1

Token2

Token3

Token4
```

Entry

$$
(i,j)
$$

measures

how strongly

token

$$
i
$$

attends to

token

$$
j.
$$

---

# <font color='orange'> 4. Why the Dot Product? </font>

The dot product measures

how similar two vectors are.

If

Query

and

Key

point in similar directions,

their dot product is large.

If they differ,

the dot product is small.

Therefore,

the dot product naturally measures

semantic similarity.

---

# <font color='orange'> 5. Step 3 — Why Divide by $\sqrt{d_k}$? </font>

Suppose

the Key vectors have dimension

$$
d_k.
$$

As

$$
d_k
$$

increases,

dot products naturally become larger.

Large values passed into the Softmax function cause it to become extremely peaked.

Example

Without scaling

```
40

38

42
```

Softmax becomes approximately

```
0

0

1
```

Almost all probability mass concentrates on a single token.

Learning becomes unstable.

To prevent this,

we divide by

$$
\boxed{
\sqrt{d_k}.
}
$$

The scaled similarity matrix becomes

$$
\boxed{
\frac{QK^T}
{\sqrt{d_k}}.
}
$$

This keeps the scores within a reasonable numerical range.

---

# <font color='orange'> 6. Why Specifically $\sqrt{d_k}$? </font>

Assume the components of the Query and Key vectors are independent, have mean zero, and variance one.

A dot product between two vectors of length \(d_k\) is the sum of \(d_k\) products. Under these assumptions,

- the expected value of the dot product is approximately zero,
- its variance grows proportionally to \(d_k\).

Consequently, the typical magnitude (standard deviation) grows like

$$
\sqrt{d_k}.
$$

Dividing by

$$
\sqrt{d_k}
$$

normalizes the scale of the scores so that it remains approximately constant as the embedding dimension increases.

This improves numerical stability and leads to more stable gradients during training.

---

# <font color='orange'> 7. Step 4 — Apply the Softmax Function </font>

The similarity scores may be

positive,

negative,

or very large.

They must be converted into probabilities.

This is done using

Softmax.

$$
\boxed{
\alpha_i
=
\frac{e^{s_i}}
{\sum_j e^{s_j}}.
}
$$

Properties

- every value lies between

0

and

1,

- all values sum to

1.

These are the attention weights.

---

# <font color='orange'> 8. Why Softmax? </font>

Softmax performs two important tasks.

First,

it converts arbitrary scores into probabilities.

Second,

it emphasizes the largest similarities while still allowing less relevant tokens to contribute.

Thus,

the model focuses primarily on the most relevant tokens,

without completely ignoring the others.

---

# <font color='orange'> 9. Step 5 — Multiply by the Values </font>

Attention weights indicate

**where**

to look.

The Values contain

**what**

information should be retrieved.

Therefore,

the final attention output is

$$
\boxed{
\text{Output}
=
\text{Attention Weights}
\times
V.
}
$$

Every output token becomes

a weighted combination of the Value vectors.

---

# <font color='orange'> 10. The Complete Attention Equation </font>

Combining all five steps gives

$$
\boxed{
\mathrm{Attention}(Q,K,V)
=
\mathrm{Softmax}
\left(
\frac{QK^T}
{\sqrt{d_k}}
\right)
V.
}
$$

This is the fundamental equation underlying every Transformer model.

---

# <font color='orange'> 11. Matrix Dimensions </font>

Suppose

- sequence length

$$
n,
$$

- embedding dimension

$$
d,
$$

- attention dimension

$$
d_k.
$$

Then

| Matrix | Shape |
|:---|:---:|
| \(Q\) | \(n\times d_k\) |
| \(K\) | \(n\times d_k\) |
| \(V\) | \(n\times d_v\) |
| \(QK^T\) | \(n\times n\) |
| Attention Weights | \(n\times n\) |
| Output | \(n\times d_v\) |

Notice that

the attention matrix compares

every token

with

every other token.

---

# <font color='orange'> 12. Numerical Example </font>

Suppose

$$
Q=
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix},
\qquad
K=
\begin{bmatrix}
1 & 1\\
0 & 1
\end{bmatrix}.
$$

First,

compute

$$
QK^T.
$$

Since

$$
K^T=
\begin{bmatrix}
1 & 0\\
1 & 1
\end{bmatrix},
$$

we obtain

$$
QK^T
=
\begin{bmatrix}
1 & 0\\
1 & 1
\end{bmatrix}.
$$

Because

$$
d_k=2,
$$

we scale by

$$
\sqrt{2}.
$$

Applying Softmax row-wise produces the attention weights.

Finally,

multiplying by

$$
V
$$

produces the updated token representations.

This same computation is performed simultaneously for every token in the sequence.

---

# <font color='orange'> 13. Computational Complexity </font>

Suppose

the sequence contains

$$
n
$$

tokens.

The matrix multiplication

$$
QK^T
$$

requires

$$
\boxed{
\mathcal O(n^2).
}
$$

This quadratic complexity is the main computational limitation of standard Transformers.

It motivates many modern variants,

including

- Longformer,
- Performer,
- Linformer,
- FlashAttention.

---

# <font color='red'> 14. Mathematical Foundations </font>

The complete scaled dot-product attention computation is

$$
\boxed{
Q=XW_Q,
}
$$

$$
\boxed{
K=XW_K,
}
$$

$$
\boxed{
V=XW_V,
}
$$

$$
\boxed{
S
=
QK^T,
}
$$

$$
\boxed{
\hat S
=
\frac{S}
{\sqrt{d_k}},
}
$$

$$
\boxed{
A
=
\mathrm{Softmax}
(\hat S),
}
$$

$$
\boxed{
Y
=
AV.
}
$$

This sequence of operations forms the core computation inside every Transformer layer.

---

# <font color='orange'> 15. Common Misconceptions </font>

### Misconception 1

> Self-attention compares every token only with its neighbours.

**False.**

Each token compares itself with **every** token in the sequence, regardless of their positions.

---

### Misconception 2

> The scaling factor is optional.

**False.**

Without scaling, the attention scores can become very large for high-dimensional vectors, causing the Softmax function to saturate and making optimization more difficult.

---

### Misconception 3

> The attention output is one of the original Value vectors.

**False.**

The output is a **weighted combination** of all Value vectors, where the weights are determined dynamically by the attention mechanism.

---

# <font color='purple'> 16. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Query (Q) | Represents what a token is searching for |
| Key (K) | Represents how a token can be matched |
| Value (V) | Contains the information to be shared |
| Similarity Matrix | Computed as \(QK^T\) |
| Scaling | Division by \(\sqrt{d_k}\) to stabilize training |
| Softmax | Converts similarity scores into attention weights |
| Attention Output | Weighted combination of the Value vectors |

> **Key Insight:** Scaled Dot-Product Attention enables every token in a sequence to dynamically gather information from all other tokens. By comparing Queries with Keys, normalizing the resulting similarity scores, and using them to combine the Value vectors, the model learns contextual representations that depend on the entire sequence. This elegant computation is the mathematical core of the Transformer architecture and underpins nearly all modern large language models.